# 02 — CoverlessStegPort Adapters & Variance Testing

Fast CPU-only workflow using precomputed SAM `.h5` files — no GPU, SAM weights, or raw COCO images required.

In [ ]:
from pathlib import Path
import numpy as np

from coverless_steg import (
    BBoxDistanceAdapter,
    CoverlessStegPort,
    SAMAdapter,
    binary_to_string,
    string_to_binary,
    test_adapter_variance,
)

DATASET_DIR = Path("dataset")
CLEAN_H5 = (
    DATASET_DIR / "SAM_outputs_clean.h5"
    if (DATASET_DIR / "SAM_outputs_clean.h5").exists()
    else DATASET_DIR / "SAM_outputs.h5"
)
SOURCES = {
    name: path
    for name, path in [
        ("clean", CLEAN_H5),
        ("noise", DATASET_DIR / "SAM_outputs_noise.h5"),
        ("compression", DATASET_DIR / "SAM_outputs_compression.h5"),
    ]
    if path.exists()
}
print("Available H5 sources:", {k: str(v) for k, v in SOURCES.items()})

## 1. Embed & Extract with `SAMAdapter`

In [ ]:
sam_adapter = SAMAdapter(CLEAN_H5, grid_size=16, tolerance=0.02, mode="count")

secret_msg = "HelloWorld"
binary_msg = string_to_binary(secret_msg)

matched_keys, window_size, num_zeros = sam_adapter.embed(binary_msg)
extracted_bits = sam_adapter.extract(matched_keys, window_size, num_zeros)
recovered_msg = binary_to_string(extracted_bits)

print(f"Secret message : {secret_msg!r} ({len(binary_msg)} bits)")
print(f"Matched keys   : {matched_keys} (window={window_size}, pad_zeros={num_zeros})")
print(f"Recovered      : {recovered_msg!r} | exact_match={extracted_bits == binary_msg}")
assert recovered_msg == secret_msg

## 2. Define a Brand-New `CoverlessStegPort` Adapter in ~10 Lines

In [ ]:
class MaskAreaAdapter(CoverlessStegPort):
    """Encode whether consecutive SAM masks grow ('1') or shrink ('0') in area."""

    def encode_entry(self, entry) -> str:
        areas = entry.get("area")
        if areas is None or len(areas) < 2:
            return ""
        return "".join("1" if d > 0 else "0" for d in np.diff(areas))


area_adapter = MaskAreaAdapter(CLEAN_H5)
keys, win, pad = area_adapter.embed(binary_msg)
assert binary_to_string(area_adapter.extract(keys, win, pad)) == secret_msg
print(f"MaskAreaAdapter matched {len(keys)} images (window={win}) and recovered {secret_msg!r}!")

## 3. Measure Bit-Error Variance Across Clean, Noise, and Compression

In [ ]:
MESSAGES = ["HelloWorld", "Coverless", "SAM Steg!", "Lazy Dev 42", "BitTest"]
reports = {}

for label, adapter in [
    ("SAMAdapter(count)", sam_adapter),
    ("SAMAdapter(binary)", SAMAdapter(CLEAN_H5, grid_size=16, tolerance=0.02, mode="binary")),
    ("BBoxDistanceAdapter", BBoxDistanceAdapter(CLEAN_H5, threshold=300.0)),
    ("MaskAreaAdapter", area_adapter),
]:
    reports[label] = test_adapter_variance(adapter, CLEAN_H5, SOURCES, MESSAGES)
    print(f"\n=== {label} ===")
    print(reports[label])

In [ ]:
for cond, summary in reports["SAMAdapter(count)"].conditions.items():
    print(f"\n[{cond}] mean_ber={summary.mean_ber:.4f}, ber_var={summary.ber_variance:.6f}")
    for m in summary.messages:
        print(
            f"  msg={m.message!r:<14} imgs={m.images_used:<2} win={m.window_size:<2} "
            f"hamming={m.hamming_distance:<3} ber={m.ber:.4f} exact={m.exact_match}"
        )